# 💬 生成式AI應用開發｜第05週｜150分鐘修訂版
## 手動對話紀錄、Streaming 與可重設聊天工具
承接第四周訊息列表 few-shot，將人工範例替換成實際 API 回答；銜接第七周最近五輪 history。
本檔另存修訂，原教材保留。先備：函式、list/dict、for、切片、role/content、函式作為參數。

> **教師版**：完整答案與觀察標準；先做離線輸入檢查，再分段開啟付費示範。

## 🎯 學習目標
1. 分清人工 few-shot 範例與實際對話 history。
2. 以手動 user/assistant 配對支援追問，每輪重送 instructions。
3. 以 reset／最近N輪控制下一輪的輸入，觀察 usage。
4. 比較 response ID 與手動 history 的狀態保存方式。
5. 接收 delta、累積完整文字、確認 completed，失敗不寫入正式紀錄。
6. 用離線檢查確認程式行為，再以實際input與回答觀察模型表現。

## 🗺️ 150分鐘有效授課流程（休息另計）
| 分鐘 | 活動 | 學生成果 |
|---|---|---|
| 0–15 | 第四周銜接與獨立請求 | 區分範例與歷史 |
| 15–40 | 手動history、兩輪追問 | 訊息組裝、完成回答配對 |
| 40–55 | response ID比較 | 說明保存方式與取捨 |
| 55–80 | 必做A：reset、最近N輪 | 離線請求與有限視窗 |
| 80–105 | Streaming事件、模擬片段 | 看懂delta與completed |
| 105–135 | 必做B：串流與失敗保護 | 成功保存與失敗測試 |
| 135–150 | 離線程式檢查、真實回答觀察與同儕覆核 | 紀錄與修正理由 |
類別、長期重要事實與互動CLI為延伸；不是本週必做門檻。
若API排障超過5分鐘，先用虛構回應完成離線程式行為檢查。
新增離線小示範與細節對照供理解，不需學生修改；仍以A、B及固定測試為核心，詳細語法／類別為選讀。

選讀僅保留本週新需求：**request參數展開、history副本／就地reset、usage計算與假client。第四周的getattr／or、集合與函式參數不重教；遇到不熟悉處可回第四周「回應檢查」「A4範例檢查」「A5固定測試」查閱。


## 1️⃣ 環境與成本
金鑰只放Colab Secrets的`OPENAI_API_KEY`；本機用環境變數，不提交或顯示金鑰。
無金鑰可完成定義與離線練習。所有付費旗標預設False，安裝套件會連線但不呼叫模型。

**首次開啟／重啟後的順序**
1. 使用自己的Notebook副本，先儲存修改；確定所有`RUN_`旗標為False，互動CLI維持註解。
2. 新runtime或缺套件時執行`%pip`安裝格。已有符合需求的套件可跳過；若升級已匯入的套件，重新啟動工作階段，再從環境格開始。
3. 回查第3週「第二節：設定API Key」：確認Secrets名稱為`OPENAI_API_KEY`，允許目前Notebook的`Notebook access`，再重跑環境格。
4. 依序執行共用定義、練習定義與離線檢查：完成A→RUN_INPUT_CHECK→完成B→RUN_FAKE_STREAM_CHECK／RUN_OFFLINE_CHECKS。學生先補TODO；不用重寫已提供的helper或測試器。
5. 離線檢查通過後，只將當前付費活動旗標改True並執行該格；完成後恢復False，再儲存。勿保留True後使用「全部執行」。

runtime重啟後，程式編輯仍可從已儲存Notebook取得，但執行產生的變數、client與本機history／抽取結果須重新建立；未另存的執行狀態不會自動恢復。設定或變更Secrets後要重跑環境格，既有client不會自動更新。

**逾時、重試與版本紀錄**
下格用`API_TIMEOUT_SECONDS = 60.0`設定網路等待，並以`max_retries=0`關閉SDK自動重試。60秒是課堂起始設定，教師須依Colab實測調整；它控制網路各階段等待，不是整段串流或模型工作的總秒數上限。
每次API入口最多送出一次請求；手動重跑另計。逾時／失敗不代表免費，使用量與帳單仍須核對，先檢查錯誤再決定是否重跑。
環境格只顯示套件版本與金鑰是否可用。課前記錄Colab實測版本，再把安裝格改成驗證過的`套件==版本`；本次尚未在Colab驗證，保留現有升級方式，不把Windows完整快照當作Colab環境。
沿用`OPENAI_MODEL`與課程模型設定，課前確認教學帳號可用性。


In [ ]:
# %pip安裝至目前kernel；已安裝可跳過，升級已匯入套件後須重啟。
# Colab runtime重新開啟時安裝SDK；本機也可使用已安裝的課堂環境。
%pip install -q --upgrade openai

In [ ]:
import os
import openai
from openai import OpenAI

# Colab 優先讀取 Secrets；本機執行時則改用既有環境變數。

try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    # 非 Colab 環境會走到這裡，不需要中斷 notebook。
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")


# 沒有金鑰仍能做離線組裝與假事件練習；付費入口才檢查 client。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
# timeout控制網路等待，不是整段串流總時間；60秒為待課前實測的起始值。
API_TIMEOUT_SECONDS = 60.0
# 每次入口不自動重試，重跑需另計請求；缺金鑰保留離線模式。
client = OpenAI(max_retries=0, timeout=API_TIMEOUT_SECONDS) if os.getenv("OPENAI_API_KEY") else None
print("本週模型：", MODEL)
print("已讀取金鑰（不顯示）" if client else "離線模式：尚未設定金鑰")

# 套件版本用於課前環境紀錄；第4週已教getattr的備援讀法。
print("OpenAI SDK版本：", getattr(openai, "__version__", "版本資訊不可用"))


### 🧭 共用程式的責任（已提供，不是練習）
| 函式／變數 | 輸入與回傳 | 在流程中的位置 |
|---|---|---|
| CHAT_INSTRUCTIONS | 固定規則str | 每輪作為instructions送出 |
| require_text／require_client | 檢查輸入／client；不合法就拋錯 | API之前，不產生請求 |
| validated_output | response→有效回答str | API之後，保存history之前 |
| create_response | 問題與可選前輪ID→response物件 | 獨立請求／ID比較，每次一個請求 |
前三個檢查函式不呼叫API。`create_response`回傳的是response物件，顯示文字要讀其output_text；後面的`ask_chat_once`才直接回傳回答str。
下列兩格按順序執行定義；不要把API helper改成學生練習答案。

In [ ]:
import json

CHAT_INSTRUCTIONS = """
# Identity
你是生成式 AI 應用開發課程的助教。
# Instructions
- 使用繁體中文，先直接回答，再補充必要說明。
- 回答控制在 150 字內；不知道時清楚說明限制。
- 不要聲稱記得目前對話以外的資料。
"""



def require_text(text, name="user_input"):
    """空白輸入在本機擋下，不產生付費請求。"""
    if not isinstance(text, str) or not text.strip():
        raise ValueError(f"{name} 必須是非空白字串。")


def require_client(api_client):
    if api_client is None:
        raise RuntimeError("請設定 OPENAI_API_KEY，重新執行環境 cell。")


def validated_output(response):
    """HTTP 請求成功仍可能沒有可用答案；完成、拒答、空文字要分開檢查。"""
    if response.status != "completed":
        raise RuntimeError(f"回應未完成：{response.status}；不保存部分內容。")
    for item in response.output or []:
        for part in getattr(item, "content", []) or []:
            if getattr(part, "type", None) == "refusal":
                raise RuntimeError("模型拒答；本輪不寫入對話紀錄。")
    require_text(response.output_text, "output_text")
    return response.output_text

### `response = active_client.responses.create(**request)`（細節選讀）

**對應下一個正式code cell的create_response函式。** request是先準備好的參數字典，`**request`在呼叫時把字典展開為「參數名稱=值」。它不是把整個字典當成一個位置參數傳入，也不是在這裡做次方運算。

| request中的key | 展開後的具名參數 | 正式程式中的值 |
|---|---|---|
| "model" | model=MODEL | 指定模型 |
| "instructions" | instructions=CHAT_INSTRUCTIONS | 每輪規則 |
| "input" | input=user_input | 本次問題 |
| "store" | store=store | 呼叫端指定的保存設定 |
| "previous_response_id"（有前輪ID才加入） | previous_response_id=previous_response_id | 串接前輪回應 |

例如，沒有前輪ID時：

```python
# 以下兩種呼叫傳入的參數相同；只是閱讀對照，不需另外執行API。
response = active_client.responses.create(**request)

# 展開後相當於：
response = active_client.responses.create(
    model=MODEL,
    instructions=CHAT_INSTRUCTIONS,
    input=user_input,
    store=store,
)
```

**為什麼原程式先建立字典？** `request = dict(...)`準備共用參數；若previous_response_id有值，再加一個key。呼叫仍維持同一行，第一次請求不帶前輪ID、追問時才帶入，不必寫兩份幾乎相同的API呼叫。

注意字典key須為字串，且API必須接受該參數名稱；不是任意字典都能傳入。若同時寫`create(model=MODEL, **request)`且request已含model，會因參數重複而拋出TypeError。

**下一格離線示範與正式程式的對照：**

| 離線小示範 | 正式程式 |
|---|---|
| toy_request_parameters | request參數字典 |
| toy_show_request(**toy_request_parameters) | active_client.responses.create(**request) |
| 加入虛構previous_response_id | if previous_response_id成立時增加參數 |
| 回傳dict供assert觀察 | 正式API回傳response物件，還須validated_output檢查 |

小示範只顯示Python收到的參數，不呼叫API。先觀察第一筆前輪ID為None，再觀察加入key後變成虛構ID；回到create_response找出建立字典、條件加入、展開呼叫三個位置。

後面的假client還會用到`def create(**kwargs)`：函式定義中的`**kwargs`是**收集**具名參數成字典，呼叫端的`**request`則是**展開**字典；兩者方向相反。


In [ ]:
# 細節選讀：對照create_response的request與**request；只驗證Python參數傳遞，不連線。
def toy_show_request(model, instructions, input, store, previous_response_id=None):
    # 這裡接收的是各個具名參數，不是單一request字典。
    received = dict(model=model, instructions=instructions, input=input, store=store,
                    previous_response_id=previous_response_id)
    print("離線函式收到：", received)
    return received


# key需對應函式的參數名稱；model只是虛構字串，並未選用或呼叫真實模型。
toy_request_parameters = {
    "model": "offline-demo",
    "instructions": "只顯示收到的參數。",
    "input": "活動在哪裡？",
    "store": False,
}
# **在呼叫時展開字典：等同model="offline-demo", instructions=..., input=..., store=False。
toy_first_received = toy_show_request(**toy_request_parameters)
assert toy_first_received["input"] == "活動在哪裡？"
assert toy_first_received["previous_response_id"] is None

# 對照正式函式的條件加參數：加入key後，同一個**呼叫就會一併傳入它。
toy_request_parameters["previous_response_id"] = "fictional-previous-id"
toy_second_received = toy_show_request(**toy_request_parameters)
assert toy_second_received["previous_response_id"] == "fictional-previous-id"


In [ ]:
def create_response(user_input, previous_response_id=None, store=True, api_client=None):
    """供獨立請求與 response ID 比較示範；主線使用下方手動 history。"""
    require_text(user_input)
    active_client = client if api_client is None else api_client
    require_client(active_client)
    # 細節選讀對照：先建立具名參數字典，前輪ID存在時才加入額外key。
    request = dict(model=MODEL, instructions=CHAT_INSTRUCTIONS, input=user_input, store=store)
    if previous_response_id:
        request["previous_response_id"] = previous_response_id
    try:
        # **request把字典展開為具名參數；此處才發出一次API請求，不是將字典當單一input。
        response = active_client.responses.create(**request)
    except Exception as exc:
        raise RuntimeError(f"API 請求失敗：{exc}") from exc
    validated_output(response)
    return response

### 共用回應檢查：接回第四周

第四周request_text已示範完成狀態、拒答與非空文字檢查；第五周抽成validated_output，讓非串流與串流完成後共用，不重教getattr／or語法。
**對照ask_chat_once的兩行：** `answer = validated_output(response)` → `save_completed_turn(...)`。檢查拋錯時就不保存本輪配對；stream_response也須先驗證完成response，才能交給外層保存。


### 從第四周到第五周
| 項目 | 第四周 | 第五周 |
|---|---|---|
| assistant內容 | 人工指定範例答案 | API實際完成的回答 |
| 用途 | 示範任務模式 | 提供前文以支援追問 |
| 保存 | 固定範例 | 隨互動增加的本機history |
規則仍放instructions；看到Notebook有舊輸出，不代表模型收到它。
先預測第二輪要看到哪些訊息，再印出送出的input。

## 2️⃣ 獨立請求：為何忘記前文？
兩次請求若沒有附history或response ID，第二次看不到第一輪。
觀察是否拒絕猜測姓名；即使碰巧猜對，也不能證明具有記憶。

In [ ]:
# 獨立請求示範：先設定金鑰；2次API請求，確認後才開啟。
RUN_STATELESS_DEMO = False
if RUN_STATELESS_DEMO:
    first = create_response("我叫小安，專題是校園活動推薦。", store=False)
    second = create_response("我的姓名與專題是什麼？", store=False)
    print(first.output_text)
    print(second.output_text)

## 3️⃣ 手動history：先做最小的兩輪流程
第一輪只送user；收到有效完成回答後，保存user與assistant。
第二輪把已完成配對加上新user送入input。固定instructions每輪提供。
下列最小示範先不裁切，避免一次加入太多概念；練習A再實作有限視窗。
本週為單一文字回覆的簡化history；需要保留推理／工具項目的正式流程應保存必要的完整output項目。

### 先用離線配對看懂history（示範，不需修改）
下格用虛構的校園活動問答，不呼叫模型。它對應接下來RUN_MANUAL_DEMO中的三行：
| 小示範 | 正式示範 |
|---|---|
| toy_history | demo_history，只放已完成user/assistant配對 |
| toy_request = toy_history + [...] | request_input；`+`產生新列表，本次user暫時不寫入history |
| toy_history.extend([...]) | 完成回答驗證後才保存這一輪配對 |
先預測第二輪request有幾則，再看列印：應為3則；儲存第二輪後history才變成4則。
虛構toy_answer不是模型回答；正式示範的answer來自validated_output(response)。

In [ ]:
# 語法選讀：對照RUN_MANUAL_DEMO的request_input與demo_history；下列回答是虛構資料。
toy_history = [
    {"role": "user", "content": "活動在哪裡？"},
    {"role": "assistant", "content": "圖書館一樓。"},
]
toy_question = "需要報名嗎？"
# 用+建立本輪input，附上新user但不先改history；正式API失敗時不應留下半輪紀錄。
toy_request = toy_history + [{"role": "user", "content": toy_question}]
print("第二輪input：", toy_request, "｜原history則數：", len(toy_history))
toy_answer = "這是虛構示範：需要先報名。"
# 正式程式必須等API回答通過檢查，才做下列保存。
# extend加入兩個dict，等價於正式示範通過validated_output後保存user與assistant。
toy_history.extend([
    {"role": "user", "content": toy_question},
    {"role": "assistant", "content": toy_answer},
])
# 第二輪input為舊配對2則＋新user1則；保存成功後history才增加到4則。
assert len(toy_request) == 3 and len(toy_history) == 4
print("保存後history則數：", len(toy_history))

In [ ]:
# 手動history示範：先設定金鑰；2次API請求，確認後才開啟。
RUN_MANUAL_DEMO = False
if RUN_MANUAL_DEMO:
    require_client(client)
    demo_history = []
    for question in ["我叫小安，專題是校園活動推薦。", "請重述我的姓名與專題。"]:
        request_input = demo_history + [{"role": "user", "content": question}]
        print("本輪實際input：", request_input)
        response = client.responses.create(
            model=MODEL, instructions=CHAT_INSTRUCTIONS, input=request_input, store=False,
        )
        answer = validated_output(response)
        demo_history.extend([
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
        print(answer)

### ✍️ 預測與觀察
| 回合 | 預測input訊息數 | 實際input | 是否依前文作答 | 有無編造 |
|---|---|---|---|---|
| 第一輪 |  |  |  |  |
| 第二輪 |  |  |  |  |
判讀：第一輪1則，第二輪3則；第二輪assistant是第一輪真正收到的答案。
回答措辭可以不同，姓名／主題須符合前文。

## 4️⃣ response ID作比較（不另寫一套完整聊天工具）
| 方法 | 本機保存 | 每輪傳送 | 本週定位 |
|---|---|---|---|
| 手動history | 完成的訊息配對 | 最近N輪＋本次問題 | 核心實作、第七周銜接 |
| previous_response_id | 前輪response.id | 前輪ID＋本次問題 | 比較示範 |
| Conversations API | conversation ID | 依API管理持久對話 | 概念選讀 |
response ID串接仍每輪提供instructions；鏈中舊輸入仍可能計入input tokens。
本比較採store=True；本機reset不等於刪除平台已保存response。
store=False也不代表資料沒有傳送或處理；只使用虛構資料。

In [ ]:
# response ID比較：先確認store=True保存方式；2次API請求，確認後才開啟。
RUN_ID_COMPARISON = False
if RUN_ID_COMPARISON:
    id_first = create_response("我叫小安，專題是校園活動推薦。", store=True)
    id_second = create_response("請重述姓名與專題。", previous_response_id=id_first.id, store=True)
    print(id_first.id, "→", id_second.id)
    print(id_second.output_text)

## 5️⃣ 必做A：本機state、最近N輪與reset（55–80分）
**情境**：聊天逐漸變長，每輪只送最近N輪；使用者reset後，下一次發問不帶舊資料。
**已提供**：new_chat_state、history_size、save_completed_turn、ask_chat_once與測試器。
**學生負責**：build_chat_input與reset_chat；不用重寫API入口或自行呼叫save_completed_turn。
| 任務 | 具體工作 | 完成條件 |
|---|---|---|
| A1a | 閱讀已提供的文字／max_turns驗證；可加強history配對檢查 | max_turns須為正整數，history以完整配對為前提 |
| A1b | 選history最近2*max_turns則 | 不拆開user/assistant一輪 |
| A1c | 複製role/content到新訊息dict | 改input內容不能影響原history |
| A1d | 加本次user、return新列表 | 空history為1則；最近1輪為3則 |
| A2 | 清空原state，再以new_chat_state更新 | 原dict仍是同一物件，history／usage／成功數重設 |
**操作**：執行共用定義→完成A1/A2→重跑函式定義→RUN_INPUT_CHECK改True並重跑→通過後才開RUN_PRACTICE_A。
離線檢查0次API；付費A用兩輪追問＋reset後一輪，共3次。
`max_turns=5`最多帶10則已完成訊息＋本次user。裁切控制input，本機history仍保留全部；reset才清空本機狀態，不刪除平台紀錄或帳單。

### state欄位與資料流（共用程式，不需補TODO）
| state欄位 | 型別 | 何時改變？ |
|---|---|---|
| history | list[dict]，已完成訊息配對 | 有效回答後加入兩則；reset清空 |
| usage | dict，三個token累計 | 只累計已接受的成功回合 |
| completed_requests | int | 每個成功回合加1；reset歸零 |
下一格建立state、觀察字元數；再下一格save_completed_turn保存回答與統計。它不呼叫API，也不驗證回答，呼叫端必須先驗證。
`history_size`的字元數不是token。usage缺少時helper以0累計；不能據此推論該請求免費。
`history_size`中的`sum(len(m["content"]) for m in messages)`逐筆計算content長度後加總，可讀成一般for累計；只觀察字元規模，不需另外改寫生成器。


In [ ]:
def new_chat_state():
    """本機保存已完成 history、成功回合使用量與成功請求數。"""
    return {
        "history": [],
        "usage": {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0},
        "completed_requests": 0,
    }


def history_size(messages):
    """字元數方便觀察，不能當成精準 token 數或費用。"""
    # 細節選讀：生成器逐筆提供content字元數給sum，不建立中間列表；不是token統計。
    return {"訊息數": len(messages), "總字元數": sum(len(m["content"]) for m in messages)}

In [ ]:
def save_completed_turn(state, user_text, answer, response):
    """所有回應檢查通過後才一次寫入 user/assistant 配對。"""
    # 先準備 usage，避免格式讀取錯誤後留下只更新一半的 state。
    usage = getattr(response, "usage", None)
    # 細節選讀對照：字典生成式逐個key計算新累計；先完成計算，再一起更新history與usage。
    new_usage = {
        key: state["usage"][key] + (getattr(usage, key, 0) or 0)
        for key in state["usage"]
    }
    state["history"].extend([
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": answer},
    ])
    state["usage"] = new_usage
    state["completed_requests"] += 1

### usage如何累計？（細節選讀，對照save_completed_turn）

第四周已說明getattr／or，不重複解釋。本週新增的是new_usage字典生成式：`for key in state["usage"]`依序取得input_tokens、output_tokens、total_tokens的key，再用「原累計＋本次用量」建立新dict。
若不熟悉生成式，可讀成先建空dict，再用for逐個key設定加總結果；學生不需修改這個共用函式。
先算好new_usage，才保存user／assistant配對與更新統計，避免讀用量出錯後留下半套state。最後completed_requests加1只計成功回合；缺usage以0處理，不代表請求免費。


### A的函式規格與離線語法對照
| 函式 | 輸入 | 回傳／副作用 |
|---|---|---|
| build_chat_input(user_text, history, max_turns) | 問題str、完整配對list、正整數 | 新list[dict]；不修改history |
| reset_chat(state) | new_chat_state建立的dict | 就地修改，回傳None；不要用state = reset_chat(state) |
下一格用字串列表看切片：每輪兩項，所以最近一輪取最後2項。對應A1b；正式訊息是dict，A1c還須複製dict，只有列表切片不能隔離dict內容。
reset中的`state.clear()`與`state.update(...)`改原物件。若函式內只寫`state = new_chat_state()`，只是把區域變數指向另一個dict，外面的state不會清空。
下面只示範切片與物件關係；學生仍須自行完成正式訊息組裝及reset。
教師參考的enumerate提供索引與訊息，索引從0開始；index % 2用於判斷user／assistant交錯。這是配對驗證提示，不另設選讀示範，也可用一般for完成。


In [ ]:
# 語法選讀：對照A1最近N輪與A2就地reset；只示範資料操作，不提供完整練習答案。
toy_turns = ["問1", "答1", "問2", "答2", "問3", "答3"]
# 每輪問答兩項，-2從倒數第二項取到結尾；正式history改用2*max_turns計算。
print("最近一輪：", toy_turns[-2:], "｜原列表：", toy_turns)
# 切片產生新列表，但裡面的dict仍可能是同一物件。
toy_records = [{"content": "原值"}]
toy_shallow = toy_records[:]
# 故意改切片中的dict，觀察原列表也變更：A1c必須建立新的訊息dict才可隔離。
toy_shallow[0]["content"] = "共用dict已變更"
assert toy_records[0]["content"] == "共用dict已變更"
print("切片仍共用內部dict：", toy_records)
toy_settings = {"theme": "dark"}
# 兩個名稱指向同一dict；接下來clear／update修改該物件，所以兩個名稱都看得到變更。
toy_alias = toy_settings
toy_settings.clear()
toy_settings.update({"theme": "light"})
# 對照reset_chat：要保留原state身分，不能只在函式內把state指向另一個新dict。
assert toy_alias is toy_settings and toy_alias["theme"] == "light"
print("就地更新後，另一個引用也看得到：", toy_alias)

In [ ]:
def build_chat_input(user_text, history, max_turns=5):
    """只取最近完整N輪，再附上本次user；不修改原始history。"""
    require_text(user_text)
    if type(max_turns) is not int or max_turns <= 0:
        raise ValueError("max_turns必須是正整數。")
    if len(history) % 2:
        raise ValueError("history必須由已完成訊息配對組成。")
    # 細節選讀：enumerate從0提供索引，% 2區分一輪的user／assistant位置。
    for index, message in enumerate(history):
        expected_role = "user" if index % 2 == 0 else "assistant"
        if message.get("role") != expected_role:
            raise ValueError("history的角色順序錯誤。")
        require_text(message.get("content"), "history.content")
    recent = history[-2 * max_turns:]
    # 細節選讀：列表生成式每次建立新dict，避免input改動影響原history。
    return [{"role": m["role"], "content": m["content"]} for m in recent] + [
        {"role": "user", "content": user_text},
    ]


def reset_chat(state):
    """清空本機狀態，保留原dict物件讓呼叫端繼續使用。"""
    state.clear()
    state.update(new_chat_state())

### A完成後：沿著非串流入口讀一次
**對應下一格ask_chat_once**：build_chat_input組裝→create一次API→validated_output取得回答→save_completed_turn更新state→return answer。
build_chat_input的回傳值存入messages，才放進API input；instructions每輪重送。
新user還不在state裡，若API或回應驗證拋錯，保存函式不會執行；原state因此保留。
下一格程式完整提供，學生只補A1/A2。先跑後面的RUN_INPUT_CHECK（離線），通過後才做付費三輪。

In [ ]:
def ask_chat_once(state, user_text, max_turns=5, instructions=CHAT_INSTRUCTIONS, api_client=None):
    """手動 history 的一次問答；完成後保存，失敗不改 state。"""
    require_text(user_text)
    require_text(instructions, "instructions")
    messages = build_chat_input(user_text, state["history"], max_turns)
    active_client = client if api_client is None else api_client
    require_client(active_client)
    try:
        response = active_client.responses.create(
            model=MODEL, instructions=instructions, input=messages, store=False,
        )
    except Exception as exc:
        raise RuntimeError(f"聊天失敗，狀態未更新：{exc}") from exc
    answer = validated_output(response)
    save_completed_turn(state, user_text, answer, response)
    return answer

### A的檢查與預期結果（測試器已提供）
RUN_INPUT_CHECK預設False；完成A後改True並重跑下格，不需要金鑰。
sample_history四則，max_turns=1應保留最後兩則加本次user，共3則；sample_history仍為4則。
scratch重設後應等於new_chat_state：history空列表、usage三項0、completed_requests為0。
assert失敗先核對A1/A2，不刪assert或改預期資料。輸入副本是否與原history隔離，會在後續補充檢查中確認。
通過後開RUN_PRACTICE_A：第一輪input1則／history2則；第二輪input3則／history4則；reset後發問input1則，新session成功數從1重新累計。

In [ ]:
RUN_INPUT_CHECK = False
if RUN_INPUT_CHECK:
    sample_history = [
        {"role": "user", "content": "我叫小安"}, {"role": "assistant", "content": "你好小安"},
        {"role": "user", "content": "我喜歡Python"}, {"role": "assistant", "content": "了解"},
    ]
    trimmed = build_chat_input("我叫什麼？", sample_history, max_turns=1)
    assert trimmed == sample_history[-2:] + [{"role": "user", "content": "我叫什麼？"}]
    assert len(sample_history) == 4
    print("真正送入的input：", trimmed)
    print("規模：", history_size(trimmed))
    scratch = new_chat_state()
    scratch["history"] = sample_history.copy()
    reset_chat(scratch)
    assert scratch == new_chat_state()

In [ ]:
# 練習A：先完成A與RUN_INPUT_CHECK；3次API請求，確認後才開啟。
RUN_PRACTICE_A = False
if RUN_PRACTICE_A:
    practice_state = new_chat_state()
    for question in ["我叫小安，專題是課程FAQ。", "我的專題是什麼？"]:
        print("本輪input：", build_chat_input(question, practice_state["history"], 5))
        print(ask_chat_once(practice_state, question))
    print("累積usage：", practice_state["usage"])
    reset_chat(practice_state)
    print("reset後input：", build_chat_input("我的專題是什麼？", practice_state["history"]))
    print(ask_chat_once(practice_state, "我的專題是什麼？"))

### ✍️ 狀態觀察表
| 操作 | 本機history則數 | 實際input則數 | 成功請求數 | usage | 判讀 |
|---|---|---|---|---|---|
| 第一輪 |  |  |  |  |  |
| 第二輪 |  |  |  |  |  |
| reset後 |  |  |  |  |  |
| 裁切最近1輪 |  |  |  |  |  |
「送出的資料不含姓名」是reset／裁切的證據；模型猜對或猜錯本身不是可靠證據。
usage只累計成功回合，不代表所有帳單費用；失敗／中斷可能仍有費用，須核對平台用量。

## 6️⃣ Streaming：先模擬，再寫事件迴圈

### `stream=True`改變什麼？

Streaming（串流）讓程式在模型仍產生回答時，先接收並顯示已送達的文字片段。`stream`是API的具名參數，`True`／`False`是Python布林值；本節使用一般同步的`client.responses.create()`。

| 比較項目 | 未設定`stream`或`stream=False` | `stream=True` |
|---|---|---|
| 取得資料的方式 | 等待本次回應回傳，再取得一個response物件 | 取得可用`for`逐筆讀取的事件串流 |
| 文字在哪裡 | 本週先用`validated_output(response)`檢查，再讀回答文字 | 文字片段在`response.output_text.delta`事件的`event.delta`；完成事件另提供`event.response` |
| 如何顯示 | 取得並驗證回答後，再`print(answer)` | 收到文字事件就交給`print_delta`顯示，並累積片段 |
| 何時保存對話 | 回應完成且文字驗證通過後 | 收到`response.completed`，且完成文字與片段驗證通過後 |

**只加`stream=True`不會自動顯示文字。** 它控制API傳回資料的方式；程式仍須讀取事件、判斷類型，再把文字送到輸出位置。`stream`物件不是完成的response，不能直接用`stream.output_text`讀整段答案；事件也不全是文字，可能表示開始、完成或失敗。

### 對照本週正式程式

- 前面的`ask_chat_once()`沒有設定`stream`：`response = active_client.responses.create(...)` → `validated_output(response)` → 保存配對 → 回傳完整回答。
- 第7節的`stream_response()`設定`stream=True`：`for event in active_client.responses.create(...)`逐筆讀事件 → 文字事件交給`callback(event.delta)`顯示並累積 → 完成事件取得response並驗證 → `ask_chat_stream()`才保存配對。

這裡的`for`會等待下一個事件；`stream=True`不會把本週同步程式自動變成非同步或背景執行。它可讓使用者較早看到文字，整段回答的完成時間仍取決於模型與網路。

**先看下一格離線小示範：** `simulated_chunks`是已準備好的虛構片段，不呼叫API；觀察`print`逐片顯示、`append`保留片段，最後`join`合併。這格只示範文字處理；正式事件串流仍須依下表區分文字、完成與失敗，不能因畫面已有文字就認定成功。

### 事件與保存時機

| 事件 | 做什麼 | 是否寫入正式history |
|---|---|---|
| response.created | 知道串流開始 | 否 |
| response.output_text.delta | 顯示片段、累積文字、增加計數 | 否 |
| response.completed | 驗證完成文字、取得usage | 通過檢查後才保存 |
| error／response.failed／response.incomplete | 提示失敗、停止 | 否 |
| 結束卻沒有completed | 視為未完成 | 否 |
delta數不是token數，也不是字數。收到片段不表示整輪成功。
畫面可能已有部分文字；失敗時須標示中斷，不能把它當完整答案保存。
對照下一格：`print(delta, end="", flush=True)`連續顯示片段；`"".join(collected)`依序接成字串。這兩行服務串流顯示／累積，不代表已完成或已保存。


In [ ]:
# 不付費的小示範：迴圈同時顯示與累積，不需要callback或nonlocal。
# 對照stream_response的event.delta：這裡直接用片段列表，不模擬完整API事件。
simulated_chunks = ["你好", "，", "我可以幫忙。"]
collected = []
delta_count = 0
for delta in simulated_chunks:
    # 對照print_delta：end避免每片段換行，flush要求立即送出顯示，並非立即保存history。
    print(delta, end="", flush=True)
    # 對照chunks.append(event.delta)：保留原順序，稍後join才取得完整字串。
    collected.append(delta)
    # 對照chunk_count：每個片段加1；片段數不是token數，也不是字元數。
    delta_count += 1
# 本格沒有completed事件，僅展示累積方式；正式程式還須完成與拒答檢查才能保存。
print("\n完整文字：", "".join(collected), "｜片段數：", delta_count)

### 接回第四周：把顯示函式交給串流器

第四周A5把分類函式交給測試器；本週相同概念是`on_delta=print_delta`或`on_delta=displayed.append`，收到delta才執行`callback(event.delta)`，不再另做函式物件小示範。
**對照stream_response：** callback只負責顯示／收集片段；chunks與chunk_count由事件迴圈保存及計數。正式history仍須等完成驗證後更新，不能在callback裡提前保存。


## 7️⃣ 必做B：串流事件與成功保存（105–135分）
**情境**：邊收到片段邊顯示，整輪完成且驗證通過才保存正式對話。
**已提供**：print_delta、參數驗證、chunks／chunk_count／completed_response初值、callback選擇、ask_chat_stream與假事件測試器。
**學生負責**：stream_response中的事件迴圈與完成後檢查；不改保存函式。
| TODO | 具體項目 | 完成條件 |
|---|---|---|
| B1 | 呼叫active_client.responses.create並用for讀事件 | stream=True、store=False；每次函式一個請求 |
| B2 | delta分支：累積、加1、呼叫callback | 片段依原順序保留，不在此更新state |
| B3 | completed保存event.response；error／failed／incomplete拋錯 | 非文字事件如created可忽略 |
| B4 | 迴圈結束必須有completed_response，交validated_output驗證 | 缺完成、拒答、空文字都不能成功 |
| B5 | join chunks與完成文字比較，回傳三個值 | (完整str, response物件, delta數int)，文字不一致拋錯 |
請用try/except把請求、事件迭代及callback的失敗轉為RuntimeError，讓呼叫端一致辨識未成功。
**操作**：完成B1–B5並重跑定義→執行假回應／假事件定義→開RUN_FAKE_STREAM_CHECK→通過後開RUN_OFFLINE_CHECKS→最後才開付費RUN_PRACTICE_B（2次請求）。
教師可用RUN_STREAM_DEMO示範1次；學生版也依賴A/B已完成，未完成時可看前面的離線片段示範。

In [ ]:
def print_delta(delta):
    """具名 callback：只顯示片段，不負責保存正式對話紀錄。"""
    # 細節選讀：end取消自動換行，flush刷新輸出；畫面片段尚不是正式history。
    print(delta, end="", flush=True)


def stream_response(messages, instructions=CHAT_INSTRUCTIONS, on_delta=None, api_client=None):
    """手動 history 串流入口；回傳完整文字、完成response、delta數。"""
    require_text(instructions, "instructions")
    if not isinstance(messages, list) or not messages:
        raise ValueError("messages 必須是非空列表。")
    active_client = client if api_client is None else api_client
    require_client(active_client)
    chunks = []
    chunk_count = 0
    completed_response = None
    # 函式參數的呼叫方式與第四周的測試器相同；不需要 nonlocal。
    # 沿用第四周函式參數：此處選擇函式物件，收到delta時才呼叫callback。
    callback = print_delta if on_delta is None else on_delta

    try:
        for event in active_client.responses.create(
            model=MODEL, instructions=instructions, input=messages, store=False, stream=True,
        ):
            if event.type == "response.output_text.delta":
                # 計數在同一層迴圈，與 callback、模型 token 數分開。
                chunks.append(event.delta)
                chunk_count += 1
                callback(event.delta)
            elif event.type == "response.completed":
                completed_response = event.response
            elif event.type in {"error", "response.failed", "response.incomplete"}:
                raise RuntimeError(f"串流失敗或未完成：{event.type}")
    except Exception as exc:
        raise RuntimeError(f"串流未成功，狀態不得更新：{exc}") from exc
    if completed_response is None:
        raise RuntimeError("串流結束卻沒有 completed 事件。")
    answer = validated_output(completed_response)
    full_text = "".join(chunks)
    # 本週僅處理單一文字回答；確認畫面片段與完成回答一致才接受。
    if full_text != answer:
        raise RuntimeError("文字片段與完成回答不一致，請重新確認回應。")
    return full_text, completed_response, chunk_count

### B的回傳值與保存位置（對照stream_response→ask_chat_stream）
| 層次 | 回傳 | 是否修改state？ |
|---|---|---|
| stream_response | full_text、完成response、chunk_count三個值 | 不接收state，不保存history |
| ask_chat_stream | full_text、chunk_count兩個值 | 驗證成功後呼叫save_completed_turn |
| 執行區text, count = ask_chat_stream(...) | 接收文字與片段數，顯示觀察 | 使用stream_state保存兩輪 |
下一格的`full_text, response, chunk_count = stream_response(...)`是三值解包；若B只回傳str，這一行就不符合規格。
發生例外時不會進入下一行save_completed_turn，history／usage／成功數不變；但callback已顯示或收集的片段可能存在，畫面需標示中斷。
程式核對完整文字不等於查證內容正確，姓名／專題仍需人工核對。


In [ ]:
def ask_chat_stream(state, user_text, max_turns=5, instructions=CHAT_INSTRUCTIONS,
                    on_delta=None, api_client=None):
    """重用非串流的訊息組裝及成功保存邏輯，只有取得回答的方式不同。"""
    messages = build_chat_input(user_text, state["history"], max_turns)
    full_text, response, chunk_count = stream_response(
        messages, instructions, on_delta=on_delta, api_client=api_client,
    )
    save_completed_turn(state, user_text, full_text, response)
    return full_text, chunk_count

In [ ]:
# 串流示範：先完成A/B；1次API請求，確認後才開啟。
RUN_STREAM_DEMO = False
if RUN_STREAM_DEMO:
    demo_state = new_chat_state()
    demo_text, demo_count = ask_chat_stream(demo_state, "用兩句話說明API。")
    print("\n完整文字：", demo_text, "delta數：", demo_count)
    print("正式history：", demo_state["history"])


In [ ]:
# 練習B：先完成A/B及離線檢查；2次API請求，確認後才開啟。
RUN_PRACTICE_B = False
if RUN_PRACTICE_B:
    stream_state = new_chat_state()
    for question in ["我叫小安，專題是課程FAQ。", "請重述我的姓名與專題。"]:
        print("本輪input：", build_chat_input(question, stream_state["history"]))
        text, count = ask_chat_stream(stream_state, question)
        print("\n片段數：", count, "｜完整字元數：", len(text))
    print("usage：", stream_state["usage"])

## 8️⃣ 驗證聊天功能：離線程式檢查與真實 API 回答觀察

### 本單元要完成什麼？

完成A／B後，先確認你寫的聊天程式能正確組裝、清空及保存對話，再用真實API觀察回答是否符合前文。完成本單元後，你應能從測試證據指出問題出在「程式處理資料」還是「模型回答內容」。

**為什麼不能只看AI有沒有答對？** 假設第一輪說「我叫小安」，第二輪問「我叫什麼？」。就算AI回答「小安」，仍須確認第二輪的`input`真的帶入前文；清空後若AI碰巧猜中姓名，也不能因此判定`reset_chat()`失效。

| 驗證階段 | 要回答的問題 | 使用資料與費用 | 完成證據 |
|---|---|---|---|
| 1：離線程式檢查 | A／B是否遵守前文帶入、reset、最近N輪、空白防護、完成才保存等行為規則？ | 固定假回應／假事件；0次真實API | 一筆假串流檢查通過，五項檢查全部PASS |
| 2：真實回答觀察 | 在程式實際送出的資料下，模型回答是否符合姓名、專題與清空後的資訊範圍？ | 真實API；啟動`RUN_LIVE_TESTS`會有3次請求 | 保存三輪實際`input`與回答，人工註記符合／不符合／未執行 |

「行為規則」在測試函式註解中也稱為「契約」：指程式應遵守的輸入、輸出與狀態更新要求。五項PASS表示這組固定案例通過，不代表程式所有情況或模型品質都已驗證。

### 老師與學生各做什麼？

- **老師**：用教師版示範如何讀測試證據，指出每項檢查對應A／B哪個功能；真實API示範的紀錄標為教師示範。
- **學生**：完成A1／A2與B1–B5，重新執行修改過的函式定義；執行已提供的測試，失敗就回A／B修正，最後填寫本單元紀錄表。
- 測試helper與測試器已完整提供，學生不需重寫；保留固定問題與檢查條件，不能刪除`assert`或改預期資料讓測試通過。

### 執行順序（每個旗標只開目前那一格）

1. 執行下一格三個假資料helper的定義；這一步只準備測試資料。
2. 開啟`RUN_FAKE_STREAM_CHECK`，執行一筆成功串流檢查，完成後恢復False。
3. 開啟`RUN_OFFLINE_CHECKS`，執行五項行為檢查，完成後恢復False。
4. 全部離線檢查通過且有可用金鑰後，依「階段2」說明開啟`RUN_LIVE_TESTS`；完成後恢復False。暫無金鑰時填「未執行／待補」。

若出現`NotImplementedError`，表示A／B仍有未完成骨架；先完成並重跑定義，再回本單元。不要只把函式定義格執行成功當作測試通過。


### 準備：用固定假資料檢查自己寫的聊天函式

下一格提供`fake_response()`、`make_fake_client()`與`make_fake_events()`。它們用預先設定的資料取代真實SDK回傳，讓每次測試都能重現成功與中斷，不必為除錯呼叫模型。

**實際執行的仍是你的A／B函式。** 呼叫時傳入`api_client=fake`，`ask_chat_once()`或`ask_chat_stream()`會改用假入口；假入口把收到的參數記在`requests`，再交回固定資料。因此可以核對「程式原本準備送什麼」和「接到回應後如何更新state」。這些紀錄沒有送到網路。

| 已提供helper | 模擬什麼 | 用來檢查什麼 |
|---|---|---|
| `fake_response()` | 完成狀態、文字、拒答欄位與虛構usage | 正式回應驗證與成功保存流程 |
| `make_fake_client(payload, requests)` | `responses.create()`入口，收到參數就記錄，再回傳payload | `input`、`instructions`、`stream`等參數是否正確 |
| `make_fake_events(kind)` | 文字片段與完成／失敗事件；`missing`故意缺少完成事件 | B是否累積片段、確認完成並拒絕中斷回合 |

**例子：** 測試器可讓假入口兩次都回「姓名是小安」。這句話是教師預先設定的，不是模型讀懂問題的成果；第1項檢查真正核對的是第二次`input`有沒有上一輪問答。

#### 如何讀下一格程式（細節選讀）

第四周已教`SimpleNamespace`的屬性讀取；這裡以它提供正式函式會讀取的欄位。`make_fake_client()`內的`create`保留對`payload`與`requests`的引用，因此外層回傳後仍能記錄參數並回傳資料，學生不用重寫閉包。

`def create(**kwargs)`將具名參數收集成dict，與第1節`**request`的展開方向相反；`requests.append(kwargs)`只是記錄，不是真實請求。假usage不能用來計算真實帳單。閱讀時對照`validated_output()`、`stream_response()`與下方`run_offline_checks()`的欄位使用。


In [ ]:
# 細節選讀：對照validated_output、stream_response與run_offline_checks；以下都是離線虛構物件。
from types import SimpleNamespace


def fake_response(text="你好！", status="completed", refusal=False):
    """虛構回應只檢查程式流程，無法驗證模型品質或真實SDK行為。"""
    # 建立output/content/type結構，讓validated_output能沿正式欄位路徑檢查拒答。
    content = [SimpleNamespace(type="refusal")] if refusal else []
    return SimpleNamespace(
        id="offline-response", status=status, output_text=text,
        output=[SimpleNamespace(content=content)],
        usage=SimpleNamespace(input_tokens=10, output_tokens=3, total_tokens=13),
    )


def make_fake_client(payload, requests):
    # 用具名create函式替代SDK請求入口；不需要建立類別或使用lambda／nonlocal。
    def create(**kwargs):
        # **kwargs接收正式入口送來的model／instructions／input等具名參數；只記錄，不連線。
        requests.append(kwargs)
        return payload
    # create沒有括號，放入函式物件；正式程式呼叫fake.responses.create(...)才執行上方函式。
    return SimpleNamespace(responses=SimpleNamespace(create=create))


def make_fake_events(kind="completed"):
    events = [
        SimpleNamespace(type="response.created"),
        SimpleNamespace(type="response.output_text.delta", delta="你好"),
        SimpleNamespace(type="response.output_text.delta", delta="！"),
    ]
    # 對照串流B3/B4：成功事件攜帶完整response；missing不補完成事件，用來測未完成保護。
    if kind == "completed":
        events.append(SimpleNamespace(type="response.completed", response=fake_response()))
    elif kind != "missing":
        events.append(SimpleNamespace(type=kind))
    return events

### 階段1A：先檢查一筆成功串流（0次真實API）

**目的：** 先確認A的訊息組裝與B的串流處理能一起工作，再檢查其他邊界情況。下一格使用你的`ask_chat_stream()`，但回應來自固定假事件。

**操作：** 確認A／B及三個假資料helper已重新定義，將下一格`RUN_FAKE_STREAM_CHECK`改True並執行；結束後改回False。

| 檢查資料 | 預期觀察 | 代表什麼 |
|---|---|---|
| `shown` | `["你好", "！"]` | B把兩個片段依序交給callback；這次callback是`shown.append`，只收集、不逐片印出 |
| `fake_text`／`fake_count` | 完整文字「你好！」／片段數2 | 片段累積與計數正確 |
| `fake_state` | history新增一組user／assistant，成功數1 | 完成後才保存本輪對話 |
| `fake_requests` | 1筆參數紀錄，`stream=True`、`store=False` | 只呼叫一次假入口，設定符合B1 |

**成功訊號：** 所有`assert`通過，最後印出「假串流：你好！」及片段數、history。這只代表固定成功案例通過，不表示真實模型或畫面逐片顯示已驗證。

`assert`用來核對實際值與預期值；不符合就拋出`AssertionError`並停止。若失敗，先看A／B是否完成且已重跑定義，再依欄位回查A1、B1–B5或成功保存流程；不修改本格預期資料。


In [ ]:
RUN_FAKE_STREAM_CHECK = False
if RUN_FAKE_STREAM_CHECK:
    fake_requests = []
    shown = []
    fake_state = new_chat_state()
    fake_client = make_fake_client(make_fake_events(), fake_requests)
    fake_text, fake_count = ask_chat_stream(
        fake_state, "離線問候", on_delta=shown.append, api_client=fake_client,
    )
    assert fake_text == "你好！" and fake_count == 2 and shown == ["你好", "！"]
    assert len(fake_state["history"]) == 2 and fake_state["completed_requests"] == 1
    assert len(fake_requests) == 1
    assert fake_requests[0]["stream"] is True and fake_requests[0]["store"] is False
    print("假串流：", fake_text, "｜delta數：", fake_count, "｜正式history：", fake_state["history"])

### 階段1B：五項離線行為檢查（0次真實API）

**目的：** 用固定案例檢查對話資料如何傳入、裁切、清空及保存。下一格`run_offline_checks()`會實際呼叫你的A／B函式；學生只需開啟`RUN_OFFLINE_CHECKS`並執行，結束後恢復False。

| 檢查 | 測試怎麼做 | 必須成立的規則與程式證據 | 失敗時回查 |
|---|---|---|---|
| 1：追問帶入前文 | 先說姓名，再問姓名 | `requests[1]["input"]`包含第一輪user／assistant與本輪user；`instructions`每輪相同 | A1組裝順序、`ask_chat_once()` |
| 2：清空對話 | 執行`reset_chat()`，再組裝新問題 | `new_input`只有新user，成功數與usage清零；此步不再發問 | A2就地更新state |
| 3：最近N輪 | 準備兩輪history，設定`max_turns=1` | 新input為最後一輪完整配對＋本次user；核對`history[-2:]` | A1b–A1d |
| 4：擋下空白 | 傳入只含空白的問題 | 拋出`ValueError`，假入口紀錄數不增加，history仍空白 | `require_text()`、呼叫前驗證 |
| 5：完成才保存 | 一次成功串流，再模擬error／failed／incomplete／缺完成事件 | 成功累積兩個片段並保存一組配對；每種失敗皆拋錯，`state == snapshot` | B2–B5與`ask_chat_stream()`保存時機 |

**如何判讀：** 全部檢查完成後才回傳並印出五行PASS。若中途出錯，整格會停止，可能一行PASS都沒印出；先看錯誤訊息與指向的`assert`／函式，再用上表定位，不能直接假設第一項失敗。

- `AssertionError`：實際結果與測試預期不符，回A／B修正並重跑定義。
- `NotImplementedError`：A／B骨架未完成。
- `NameError`：缺少前面的定義，按本單元順序重新執行。
- 測試中故意造成的`ValueError`／`RuntimeError`已由測試器接住；能正確拒絕錯誤輸入或中斷，本來就是應通過的行為。

**讀測試器時注意：** `requests`只是本機參數紀錄，不能當作帳單；`snapshot`用JSON往返建立獨立state副本，供失敗前後比較，沒有永久存檔。`try/except/else`中的`else`只在沒有例外時執行，因此測試器會把「應該拋錯卻成功」判定失敗。

五行PASS是這五組固定行為案例通過的證據，不是AI回答正確率，也不是完整品質評分。下一階段才使用真實API觀察回答內容。


In [ ]:
def run_offline_checks():
    """完成練習後先驗證五個契約；不使用 API key，也不付費。"""
    rows = []
    requests = []
    state = new_chat_state()
    fake = make_fake_client(fake_response("姓名是小安"), requests)
    ask_chat_once(state, "我叫小安", api_client=fake)
    ask_chat_once(state, "我叫什麼？", api_client=fake)
    assert requests[1]["input"] == state["history"][:3]
    assert requests[1]["instructions"] == CHAT_INSTRUCTIONS
    rows.append("PASS：第二輪包含真實保存的前文")

    reset_chat(state)
    new_input = build_chat_input("我叫什麼？", state["history"])
    assert new_input == [{"role": "user", "content": "我叫什麼？"}]
    assert state["completed_requests"] == 0 and state["usage"]["total_tokens"] == 0
    rows.append("PASS：reset 後請求沒有舊姓名")

    history = [
        {"role": "user", "content": "舊問題"}, {"role": "assistant", "content": "舊答案"},
        {"role": "user", "content": "新問題"}, {"role": "assistant", "content": "新答案"},
    ]
    assert build_chat_input("追問", history, 1) == history[-2:] + [{"role": "user", "content": "追問"}]
    rows.append("PASS：最近N輪保持完整配對")

    before = len(requests)
    try:
        ask_chat_once(state, " ", api_client=fake)
    except ValueError:
        pass
    # 細節選讀：else只在try沒有例外時執行；此測試預期空白會拋錯。
    else:
        raise AssertionError("空白輸入沒有被擋下。")
    assert len(requests) == before and state["history"] == []
    rows.append("PASS：空白輸入不呼叫API")

    displayed = []
    stream_fake = make_fake_client(make_fake_events(), requests)
    text, count = ask_chat_stream(state, "你好", on_delta=displayed.append, api_client=stream_fake)
    assert text == "你好！" and count == 2 and "".join(displayed) == text
    assert len(state["history"]) == 2
    # 細節選讀：JSON往返為本教材的簡單state建立獨立副本；不是任意物件的通用深複製。
    snapshot = json.loads(json.dumps(state))
    for kind in ("error", "response.failed", "response.incomplete", "missing"):
        bad = make_fake_client(make_fake_events(kind), requests)
        try:
            ask_chat_stream(state, "失敗回合", on_delta=displayed.append, api_client=bad)
        except RuntimeError:
            pass
        else:
            raise AssertionError(f"未拒絕：{kind}")
        assert state == snapshot
    rows.append("PASS：串流成功才保存；失敗／缺完成事件維持原狀態")
    return rows


RUN_OFFLINE_CHECKS = False
if RUN_OFFLINE_CHECKS:
    for row in run_offline_checks():
        print(row)

### 階段2：觀察真實 API 回答（啟動後3次請求）

**目的：** 離線檢查通過後，確認真實模型在收到正確資料時如何回答。下一格`run_live_chat_checks()`使用非串流的`ask_chat_once()`印出input與回答；這是三輪對話觀察，不是完整模型評測或真實串流驗證。

**先備與操作：** 完成A／B與兩項離線檢查、確認金鑰及client可用，再將下一格`RUN_LIVE_TESTS`改True並執行一次。結束後恢復False；重跑另計費用。沒有金鑰或尚未執行時填「未執行／待補」，保留離線成果。

| 輪次 | 程式送什麼 | 人工核對什麼 |
|---|---|---|
| 1：提供資料 | 「我叫小安，專題是課程FAQ。」 | 回答是否符合提供資料，是否添加未提供的個人資訊 |
| 2：追問 | 上一輪user／assistant配對＋「我的姓名與專題是什麼？」 | 印出的input確實有前文；回答姓名與專題是否相符 |
| 3：reset後再問 | 清空後只有「我的姓名是什麼？」 | input沒有舊姓名；合理回答應表示資料不足或請使用者提供姓名 |

**先看input，再看回答。** input少前文或reset後仍有舊資料，就回A／保存流程修正；input符合規格但回答不符，則記錄模型回答問題，再檢查instructions或後續重新測試。若發生連線、帳號、模型存取或回應驗證錯誤，先記錄API／回應錯誤，不能當作模型內容答錯。

reset後即使模型碰巧猜中「小安」，也不能只靠該回答判定reset失效；以實際input確認舊資料有沒有送出，並記錄回答是否無依據聲稱記得。不要用「答案一定不同」作為測試條件。

保存三輪實際input、回答與人工判讀。這些結果才是真實API觀察；固定假回應不能替代。教材usage只統計成功回合，reset後最後印出的usage也只代表清空後的累積值，不是三次請求的完整帳單；查費用仍需核對平台紀錄。


In [ ]:
def run_live_chat_checks():
    """三次真實請求，印出請求與回答供人工覆核，不用猜測式答案assert。"""
    state = new_chat_state()
    for question in ["我叫小安，專題是課程FAQ。", "我的姓名與專題是什麼？"]:
        print("input：", build_chat_input(question, state["history"]))
        print(ask_chat_once(state, question))
    reset_chat(state)
    print("reset後input：", build_chat_input("我的姓名是什麼？", state["history"]))
    print(ask_chat_once(state, "我的姓名是什麼？"))
    print("reset後usage：", state["usage"])

# 真實測試：先完成A與離線檢查；3次API請求，確認後才開啟。
RUN_LIVE_TESTS = False
if RUN_LIVE_TESTS:
    run_live_chat_checks()

### 🧪 完成紀錄：分開程式檢查與真實回答

保留測試輸出或錯誤訊息，再填寫下表。「通過」須有對應證據；真實API未執行時如實標記。老師示範的結果請標「教師示範」，學生自己的執行標「學生驗證」。

| 階段／項目 | 應保留的證據 | 實際結果（通過／不符合／未執行） | 修正位置與重測結果 |
|---|---|---|---|
| 離線1A：成功串流 | 假串流完整文字、片段數、history與參數檢查 |  |  |
| 離線1B-1：前文 | 第1項PASS；若失敗，保留錯誤位置 |  |  |
| 離線1B-2：reset | 第2項PASS／錯誤位置 |  |  |
| 離線1B-3：最近N輪 | 第3項PASS／錯誤位置 |  |  |
| 離線1B-4：空白 | 第4項PASS／錯誤位置 |  |  |
| 離線1B-5：串流中斷 | 第5項PASS／錯誤位置 |  |  |
| 真實第1輪：提供資料 | 實際input、回答、人工判讀 |  |  |
| 真實第2輪：追問 | 實際input含前文，回答姓名／專題是否相符 |  |  |
| 真實第3輪：reset後 | 實際input不含舊姓名，回答是否依據不足 |  |  |

**本單元完成條件：** 能指出五項檢查各驗證什麼，完成離線1A與五項PASS，並記錄真實三輪的觀察或待補狀態。只貼「AI回答小安」或「全部PASS」而說不出資料與狀態的檢查意義，仍須補充說明。

**同儕覆核：** 用一項測試說明「程式原本準備送什麼」或「何時更新state」；再指出真實API有無執行、input是否符合規格，及回答內容的判讀理由。


## 9️⃣ 課後整合：簡易CLI骨架（選做）

完成A／B及離線檢查後，才手動取消下一格最後一行`# run_console_chat()`的註解。CLI讓使用者連續輸入問題；課堂仍先用固定問題列表，避免互動輸入卡住「全部執行」。每次正常發問會呼叫付費API。

| 操作 | 使用時機 | 結果 |
|---|---|---|
| 輸入問題並按Enter | 出現「你：」輸入提示時 | 接收串流回答，完成才保存問答配對 |
| `/reset`＋Enter | 等待輸入時 | 清空本機對話與累積統計，繼續聊天 |
| `/exit`＋Enter | 等待輸入時 | 正常結束CLI，不再發問 |
| `Ctrl+C` | 本機終端機等待輸入或執行回答時 | Python收到中斷後，顯示停止提示並結束整個CLI |

**Colab／Notebook操作：** 瀏覽器輸入框中的`Ctrl+C`可能是複製，不一定送出Python中斷。等待輸入時可用`/exit`；回答正在執行時，使用該程式格的停止按鈕。環境若送出`KeyboardInterrupt`會由下方程式處理；強制停止工作階段可能直接中止，不能保證顯示自訂提示。課前須在實際教學環境試操作。

### 讀下一格：兩層例外處理負責不同事情

- **內層**`except (ValueError, RuntimeError)`：單輪輸入／API失敗，顯示訊息後可繼續輸入。
- **外層**`except KeyboardInterrupt`：使用者要求停止，結束整個函式，不把中斷當成可重試的聊天錯誤。外層`try`包含`input()`與`ask_chat_stream()`，所以兩個等待位置都能處理。
- **外層**`except EOFError`：輸入來源結束時，顯示提示後離開，避免CLI持續等待不存在的輸入。

`KeyboardInterrupt`是Python接到中斷操作時的例外，不會被一般`except Exception`接住，因此須另外處理。它不是自行監聽鍵盤的功能，也不需要背景執行緒。

若中斷發生在串流尚未完成時，`ask_chat_stream()`不會走到成功保存；已顯示的片段應視為未完成回答。停止本機等待不保證伺服器請求已取消或未計費；用量仍須核對平台紀錄。CLI結束後再次呼叫會建立新的state，沒有自動保存或恢復舊對話。

[Python KeyboardInterrupt／EOFError說明](https://docs.python.org/3.12/library/exceptions.html)


In [ ]:
def run_console_chat():
    """重用串流聊天；/exit或使用者中斷時結束CLI，不另寫聊天邏輯。"""
    state = new_chat_state()
    print("連續對話已啟動：/reset 清空、/exit 結束；本機終端機可按 Ctrl+C 停止。")
    # 外層涵蓋輸入與回答：停止要求應結束整個CLI，不進入單輪重試。
    try:
        while True:
            user_text = input("\n你：").strip()
            if user_text == "/exit":
                print("對話已結束。")
                break
            if user_text == "/reset":
                reset_chat(state)
                print("本機對話已清空。")
                continue
            try:
                ask_chat_stream(state, user_text)
                print()
            # tuple表示任一單輪錯誤；顯示後可繼續輸入，中斷例外交外層處理。
            except (ValueError, RuntimeError) as exc:
                print(f"\n本輪未成功：{exc}，可重試。")
    except KeyboardInterrupt:
        # 不再重試或保存未完成回答；收到中斷後盡快離開函式。
        print("\n已收到停止要求，對話已結束；若回答尚未完成，已顯示的片段請視為未完成。")
    except EOFError:
        # 輸入來源結束時直接離開，避免持續等待輸入。
        print("\n輸入已結束，對話已停止。")


# 完成練習並確認成本後，手動取消下一行註解；每輪都會付費。
# run_console_chat()


## 🔎 進階選讀：重要事實與類別封裝

重要偏好可另存本機state，作為清楚標示的使用者資料提供模型，不提升成可覆蓋instructions的指令；未寫檔或存資料庫，重啟後不保留，不稱永久記憶。
選做：另建preferred_language與project_topic，印出裁切後input確認仍含選定資料，說明reset是否清除它們；不要求額外付費實驗。

**下一格ManualChatSession只包裝已教函式，不另寫聊天邏輯：**

| 方法／屬性 | 對應函式版 |
|---|---|
| __init__中的self.state | new_chat_state，每個物件建立自己的state |
| ask | ask_chat_once，回傳str |
| ask_stream | ask_chat_stream，回傳文字與delta數 |
| reset | reset_chat(self.state)，回傳None |

self指目前物件；建立實例只初始化狀態，ask／ask_stream才呼叫API。類別不是本週必做，不需將A/B改寫為類別，也不增加繼承、super或nonlocal課程。


In [ ]:
class ManualChatSession:
    """選讀：state成為物件屬性，核心函式維持原樣。"""
    # 細節選讀：建立實例時初始化，每個實例各自呼叫new_chat_state；不呼叫API。
    def __init__(self, instructions=CHAT_INSTRUCTIONS, max_turns=5):
        self.instructions = instructions
        self.max_turns = max_turns
        self.state = new_chat_state()

    # 細節選讀：session.ask(...)會自動把目前實例傳給self，再轉交既有聊天函式。
    def ask(self, user_text):
        return ask_chat_once(self.state, user_text, self.max_turns, self.instructions)

    def ask_stream(self, user_text):
        return ask_chat_stream(self.state, user_text, self.max_turns, self.instructions)

    def reset(self):
        reset_chat(self.state)

# 只定義類別，不在此呼叫API。讀懂函式版後再比較self.state與普通state。

## ✅ 完成檢核與後續銜接
- [ ] 能說明人工範例、實際history與response ID的差別。
- [ ] 完成A：最近N輪與reset，印出真正送出的input。
- [ ] 完成B：delta累積、完成驗證、失敗不更新history。
- [ ] 固定instructions每輪送出，usage不等於完整帳單。
- [ ] 通過一筆假串流與五項離線行為檢查，記錄真實三輪input／回答或待補狀態。
- [ ] 能區分模型答錯、輸入／輸出不符合規格及API失敗。
第六周仍教Structured Outputs與資料抽取；第五周不提前塞入JSON Schema。
第七周把同一history放入st.session_state，最近五輪加本次user送入模型。


## 📝 課後任務與繳交規格
選課程FAQ、學習助教或校園資訊情境，沿用本週手動history函式，完成A/B。
| 繳交項目 | 具體內容 | 驗收方式 |
|---|---|---|
| Notebook | 完成A1/A2、B1–B5；保留測試器 | 函式可定義，離線檢查通過 |
| instructions | 身分、語言、回答限制、未知資料處理 | 每輪確實送出相同規則 |
| 對話紀錄 | 三輪含追問，另附reset後一輪 | 保存實際input與回答；不預填模型答案 |
| 最近N輪觀察 | 設N=1，印出輸入與本機history則數 | 完整配對＋本次user；裁切不刪原history |
| 串流與測試 | 片段、完整文字、delta數、五項PASS與其檢查意義、失敗時state比較 | 完成才保存；未完成不污染state |
| 真實測試與usage | 標示已執行／未執行、成功回合用量 | 不把假usage當真實用量，不以成功數代替帳單 |
| 150字反思 | 本機狀態、保存時機、失敗重試與修正理由 | 以實際input／測試結果支持說明 |
若暫無金鑰，先交離線成果並標明真實API未驗證；真實回答紀錄仍是待補項目，不能拿假回應冒充。
CLI、類別與重要事實為延伸，不要求全部完成。安全：使用虛構資料，不提交API key、學生個資、成績或機密。

## 👩‍🏫 教師判讀與常見錯誤
- 第四周assistant是人工標準答案，第五周是完成回答；不要把它說成模型被重新訓練。
- 最小兩輪input為1/3則；最近N輪是2N則已完成訊息加本次user。
- 裁切只影響本輪input，不會自動刪除本機完整history；reset才清空本機state。
- 前文／reset測試以實際請求為證，不能只看模型偶然回答。
- assistant只在完成、非空、非拒答時配對保存；錯誤回合不得加入history。
- 顯示片段與正式保存分開；失敗後畫面可能有半段文字，應標示未完成。
- delta數與tokens、字元數不同；本教材usage只統計成功回合，不可充當平台帳單。
- 常見錯誤：直接append本次user到state、切片少算兩則、把callback立即呼叫、將request成功等同回答有效。
- 學生若完成A/B及核心測試，不再要求從零寫繼承或nonlocal；將時間用於觀察與修正。
**第8節授課提示（135–150分）：** 約3分鐘說明目的並跑一筆假串流，7分鐘執行五項檢查／定位錯誤，5分鐘觀察三輪真實回答或展示教師已執行的紀錄，再完成判讀。API排障超時就保留離線成果及真實測試待補狀態。教師示範不能冒充學生自己的驗證紀錄。

追問學生：「哪一項證據證明前文有送出？假回答為何不能證明模型記得？串流已顯示片段卻中斷，history應如何變化？」評量重點是能連結A／B、測試證據與修正位置，不要求重寫假client或增加付費請求。


## 📚 官方文件與版本使用
- [Conversation state](https://developers.openai.com/api/docs/guides/conversation-state)
- [Streaming responses](https://developers.openai.com/api/docs/guides/streaming-responses)
- [Responses API](https://developers.openai.com/api/reference/resources/responses/methods/create)
模型沿用OPENAI_MODEL／gpt-5.4-mini；本次沒有用教學帳號驗證可用性，課前須試跑。
本修訂版與原教材並存，課堂選定一組使用，不要求學生交錯執行兩版。
- [OpenAI官方Python SDK說明](https://developers.openai.com/api/reference/python)：timeout與max_retries。
- [IPython %pip](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-pip)：目前kernel的套件安裝。
- [Colab FAQ](https://research.google.com/colaboratory/faq.html)：runtime與執行狀態。
